In [1]:
import pandas as pd
import openpyxl
from openpyxl import load_workbook
import re
import os
from sentence_transformers import SentenceTransformer, util

SCORING_EXCEL_PATH = r"C:\Users\kashi\OneDrive\Desktop\AutoEIT datasets\AutoEIT Test Files\Sample Audio Files and Transcriptions\AutoEIT Sample Transcriptions for Scoring.xlsx"
OUTPUT_SCORING_PATH = r"C:\Users\kashi\OneDrive\Desktop\AutoEIT\autoeit\output\test2_output.xlsx"

os.makedirs(os.path.dirname(OUTPUT_SCORING_PATH), exist_ok=True)
print("Setup complete")

Setup complete


In [2]:
# Load multilingual sentence transformer model
print("Loading scoring model...")
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Model loaded!")

Loading scoring model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded!


In [3]:
def clean_text(text):
    """Remove transcription notation before scoring"""
    if not text or not isinstance(text, str):
        return ""
    text = re.sub(r'\[.*?\]', '', text)  # Remove [pause], [gibberish] etc
    text = re.sub(r'\bxxx\b', '', text, flags=re.IGNORECASE)  # Remove xxx
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def score_utterance(stimulus, transcription):
    """Score 0-4 based on meaning-based rubric (Ortega, 2000)"""
    transcription = clean_text(transcription)
    stimulus = clean_text(stimulus)
    
    if not transcription:
        return 0
    
    # Check for exact match (score 4)
    if transcription.lower().strip() == stimulus.lower().strip():
        return 4
    
    # Semantic similarity for 1-3
    emb_s = model.encode(stimulus, convert_to_tensor=True)
    emb_t = model.encode(transcription, convert_to_tensor=True)
    sim = float(util.cos_sim(emb_s, emb_t))
    
    if sim >= 0.92:
        return 3
    elif sim >= 0.75:
        return 2
    elif sim >= 0.45:
        return 1
    else:
        return 0

print("Scoring function ready")

Scoring function ready


In [4]:
wb = load_workbook(SCORING_EXCEL_PATH)
print("Sheets:", wb.sheetnames)

# Check first sheet structure
ws = wb[wb.sheetnames[1]]
for row in ws.iter_rows(min_row=1, max_row=5):
    for cell in row:
        if cell.value:
            print(f"Row {cell.row}, Col {cell.column}: {str(cell.value)[:50]}")

Sheets: ['Info', '38001-1A', '38002-2A', '38004-2A', '38006-2A']
Row 1, Col 1: Sentence
Row 1, Col 2: Stimulus
Row 1, Col 3: Transcription Rater 1
Row 1, Col 4: Score
Row 2, Col 1: 1
Row 2, Col 2: Quiero cortarme el pelo (7)
Row 2, Col 3: Quiero cortarme el pelo
Row 3, Col 1: 2
Row 3, Col 2: El libro está en la mesa (7)
Row 3, Col 3: El libro está en la mesa
Row 4, Col 1: 3
Row 4, Col 2: El carro lo tiene Pedro (8)
Row 4, Col 3: El carro lo tiene Pedro
Row 5, Col 1: 4
Row 5, Col 2: El se ducha cada mañana (9)
Row 5, Col 3: El se ducha cada mañana


In [5]:
import re

def clean_stimulus(text):
    """Remove syllable count in parentheses from stimulus"""
    if not text:
        return ""
    return re.sub(r'\(\d+\)\s*$', '', str(text)).strip()

# Score all participants
wb = load_workbook(SCORING_EXCEL_PATH)

for sheet_name in wb.sheetnames[1:]:  # Skip Info tab
    ws = wb[sheet_name]
    print(f"\nScoring {sheet_name}...")
    
    scored = 0
    for row_num in range(2, 32):  # Rows 2-31
        stimulus_raw = ws.cell(row=row_num, column=2).value
        transcription = ws.cell(row=row_num, column=3).value
        
        if stimulus_raw:
            stimulus = clean_stimulus(stimulus_raw)
            score = score_utterance(stimulus, transcription)
            ws.cell(row=row_num, column=4, value=score)
            scored += 1
    
    print(f"  Scored {scored} utterances")

wb.save(OUTPUT_SCORING_PATH)
print(f"\nSaved to {OUTPUT_SCORING_PATH}")


Scoring 38001-1A...
  Scored 30 utterances

Scoring 38002-2A...
  Scored 30 utterances

Scoring 38004-2A...
  Scored 30 utterances

Scoring 38006-2A...
  Scored 30 utterances

Saved to C:\Users\kashi\OneDrive\Desktop\AutoEIT\autoeit\output\test2_output.xlsx


In [6]:
# Check scores look reasonable
wb_out = load_workbook(OUTPUT_SCORING_PATH)
ws = wb_out["38001-1A"]

print("Sample scores for 38001-1A:\n")
for row_num in range(2, 12):
    stimulus = clean_stimulus(ws.cell(row=row_num, column=2).value)
    transcription = ws.cell(row=row_num, column=3).value
    score = ws.cell(row=row_num, column=4).value
    print(f"Score {score} | S: {stimulus[:40]}")
    print(f"         | T: {str(transcription)[:40]}")
    print()

Sample scores for 38001-1A:

Score 4 | S: Quiero cortarme el pelo
         | T: Quiero cortarme el pelo

Score 4 | S: El libro está en la mesa
         | T: El libro está en la mesa

Score 4 | S: El carro lo tiene Pedro
         | T: El carro lo tiene Pedro

Score 4 | S: El se ducha cada mañana
         | T: El se ducha cada mañana

Score 3 | S: ¿Qué dice usted que va a hacer hoy?
         | T: Que dices ustedes se que van a hacer hoy

Score 2 | S: Dudo que sepa manejar muy bien
         | T: Dudo que sepa manajar bien

Score 3 | S: Las calles de esta ciudad son muy anchas
         | T: Las calles de esta cuidad son muy anchas

Score 1 | S: Puede que llueva mañana todo el día
         | T: Puede que lleva mañana todo el día

Score 2 | S: Las casas son muy bonitas pero caras
         | T: Las casas son muy bonitas pero muy cadas

Score 1 | S: Me gustan las películas que acaban bien
         | T: Me gustan las peliculas que acaban bien



In [7]:
import unicodedata

def normalize_text(text):
    """Remove accents for comparison"""
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    ).lower().strip()

def score_utterance(stimulus, transcription):
    transcription_clean = clean_text(transcription)
    stimulus_clean = clean_text(stimulus)
    
    if not transcription_clean:
        return 0
    
    # Exact match (score 4)
    if normalize_text(transcription_clean) == normalize_text(stimulus_clean):
        return 4
    
    # Semantic similarity
    emb_s = model.encode(stimulus_clean, convert_to_tensor=True)
    emb_t = model.encode(transcription_clean, convert_to_tensor=True)
    sim = float(util.cos_sim(emb_s, emb_t))
    
    if sim >= 0.92:
        return 3
    elif sim >= 0.75:
        return 2
    elif sim >= 0.45:
        return 1
    else:
        return 0

print("Updated scoring function ready")

Updated scoring function ready


In [10]:
# Check scores look reasonable
wb_out = load_workbook(OUTPUT_SCORING_PATH)
ws = wb_out["38001-1A"]

print("Sample scores for 38001-1A:\n")
for row_num in range(2, 12):
    stimulus = clean_stimulus(ws.cell(row=row_num, column=2).value)
    transcription = ws.cell(row=row_num, column=3).value
    score = ws.cell(row=row_num, column=4).value
    print(f"Score {score} | S: {stimulus[:40]}")
    print(f"         | T: {str(transcription)[:40]}")
    print()

Sample scores for 38001-1A:

Score 4 | S: Quiero cortarme el pelo
         | T: Quiero cortarme el pelo

Score 4 | S: El libro está en la mesa
         | T: El libro está en la mesa

Score 4 | S: El carro lo tiene Pedro
         | T: El carro lo tiene Pedro

Score 4 | S: El se ducha cada mañana
         | T: El se ducha cada mañana

Score 3 | S: ¿Qué dice usted que va a hacer hoy?
         | T: Que dices ustedes se que van a hacer hoy

Score 2 | S: Dudo que sepa manejar muy bien
         | T: Dudo que sepa manajar bien

Score 3 | S: Las calles de esta ciudad son muy anchas
         | T: Las calles de esta cuidad son muy anchas

Score 1 | S: Puede que llueva mañana todo el día
         | T: Puede que lleva mañana todo el día

Score 2 | S: Las casas son muy bonitas pero caras
         | T: Las casas son muy bonitas pero muy cadas

Score 1 | S: Me gustan las películas que acaban bien
         | T: Me gustan las peliculas que acaban bien



In [9]:
def word_overlap_ratio(s1, s2):
    """Calculate what fraction of stimulus words appear in transcription"""
    s1_words = set(normalize_text(s1).split())
    s2_words = set(normalize_text(s2).split())
    if not s1_words:
        return 0
    return len(s1_words & s2_words) / len(s1_words)

def score_utterance(stimulus, transcription):
    transcription_clean = clean_text(transcription)
    stimulus_clean = clean_text(stimulus)
    
    if not transcription_clean:
        return 0
    
    # Exact match (score 4)
    if normalize_text(transcription_clean) == normalize_text(stimulus_clean):
        return 4
    
    # Word overlap
    overlap = word_overlap_ratio(stimulus_clean, transcription_clean)
    
    # Semantic similarity
    emb_s = model.encode(stimulus_clean, convert_to_tensor=True)
    emb_t = model.encode(transcription_clean, convert_to_tensor=True)
    sim = float(util.cos_sim(emb_s, emb_t))
    
    # Combine both signals
    if sim >= 0.92 or overlap >= 0.90:
        return 3
    elif sim >= 0.75 or overlap >= 0.70:
        return 2
    elif sim >= 0.45 or overlap >= 0.40:
        return 1
    else:
        return 0

print("Updated scoring function ready")

Updated scoring function ready


In [11]:
wb = load_workbook(SCORING_EXCEL_PATH)

for sheet_name in wb.sheetnames[1:]:
    ws = wb[sheet_name]
    print(f"Scoring {sheet_name}...")
    
    for row_num in range(2, 32):
        stimulus_raw = ws.cell(row=row_num, column=2).value
        transcription = ws.cell(row=row_num, column=3).value
        
        if stimulus_raw:
            stimulus = clean_stimulus(stimulus_raw)
            score = score_utterance(stimulus, transcription)
            ws.cell(row=row_num, column=4, value=score)

wb.save(OUTPUT_SCORING_PATH)
print("Done! Saved to", OUTPUT_SCORING_PATH)

Scoring 38001-1A...
Scoring 38002-2A...
Scoring 38004-2A...
Scoring 38006-2A...
Done! Saved to C:\Users\kashi\OneDrive\Desktop\AutoEIT\autoeit\output\test2_output.xlsx


In [12]:
wb_out = load_workbook(OUTPUT_SCORING_PATH)
ws = wb_out["38001-1A"]

print("Sample scores for 38001-1A:\n")
for row_num in range(2, 12):
    stimulus = clean_stimulus(ws.cell(row=row_num, column=2).value)
    transcription = ws.cell(row=row_num, column=3).value
    score = ws.cell(row=row_num, column=4).value
    print(f"Score {score} | S: {stimulus[:40]}")
    print(f"         | T: {str(transcription)[:40]}")
    print()

Sample scores for 38001-1A:

Score 4 | S: Quiero cortarme el pelo
         | T: Quiero cortarme el pelo

Score 4 | S: El libro está en la mesa
         | T: El libro está en la mesa

Score 4 | S: El carro lo tiene Pedro
         | T: El carro lo tiene Pedro

Score 4 | S: El se ducha cada mañana
         | T: El se ducha cada mañana

Score 3 | S: ¿Qué dice usted que va a hacer hoy?
         | T: Que dices ustedes se que van a hacer hoy

Score 2 | S: Dudo que sepa manejar muy bien
         | T: Dudo que sepa manajar bien

Score 3 | S: Las calles de esta ciudad son muy anchas
         | T: Las calles de esta cuidad son muy anchas

Score 2 | S: Puede que llueva mañana todo el día
         | T: Puede que lleva mañana todo el día

Score 2 | S: Las casas son muy bonitas pero caras
         | T: Las casas son muy bonitas pero muy cadas

Score 4 | S: Me gustan las películas que acaban bien
         | T: Me gustan las peliculas que acaban bien



## Methodology & Approach

### Scoring System
Implemented the meaning-based EIT rubric (Ortega, 2000) with scores 0-4:
- **4**: Exact or near-exact repetition (normalized for accents)
- **3**: Full meaning preserved, minor changes acceptable
- **2**: More than half of meaning preserved, some inexactness
- **1**: About half of meaning, significant information missing
- **0**: No response, gibberish, or only 1-2 words

### Technical Approach
- Multilingual sentence embeddings (paraphrase-multilingual-MiniLM-L12-v2)
  used for semantic similarity scoring
- Word overlap ratio combined with semantic similarity for robustness
- Accent normalization applied before exact match check
- Special notation ([pause], [gibberish], xxx) stripped before scoring

### Threshold Calibration
Thresholds were calibrated by comparing model output against the 
provided example transcriptions in the scoring sheet, then adjusted 
to better match the rubric criteria.

### Known Limitations
- Semantic similarity models may not perfectly capture all meaning 
  distinctions relevant to L2 Spanish assessment
- Edge cases (synonymous substitutions, grammar changes that don't 
  affect meaning) are difficult to handle automatically
- Human validation against gold standard scores would be needed for 
  production use